# Speech-to-text through the AITTA API

In this notebook you send recordings to a Whisper model running on CSC's
**AITTA** service, through its **OpenAI-compatible API**, and compare the
model's transcripts with the correct texts.

0. Download the data, if it isn't there yet. The notebook needs nothing else
   from the repository.
1. Transcribe **one recording** and look at the result.
2. Transcribe **a few recordings** (at most 50) and calculate the word error rate.
3. For **all recordings** (about 900), submit the batch job `aitta/transcribe_all.sh`.
   It runs without the notebook and continues from the results this notebook saves.

The model runs on AITTA's GPUs, so this notebook needs no GPU.

## 1. Settings

`HF_HOME` tells Hugging Face where to store downloads. It is set here, before
anything is imported, so that the data does not fill your small home folder.

In [ ]:
import os

WORK = "/scratch/project_462001302/asr-data"   # data and results go here (on your own computer e.g. "~/asr-data")
LANGUAGE = "fi"                                # "fi" or "en"
SPLIT = "test"                                 # "train", "validation" or "test"

AITTA_BASE_URL = "https://aitta-api.csc.fi/openai/v1"   # AITTA's OpenAI-compatible API
MODEL = "openai/whisper-large-v3"                        # the speech-to-text model on AITTA

WORK = os.path.expanduser(WORK)
DATA = f"fleurs-{LANGUAGE}"
os.environ["HF_HOME"] = os.path.join(WORK, "hf-cache")

## 2. Imports

In [ ]:
import io
import json
import time
import wave
from getpass import getpass

import openai
import pandas as pd
import pyarrow.parquet as pq
from datasets import Dataset, Features, Value, load_from_disk
from huggingface_hub import HfApi, snapshot_download
from IPython.display import Audio

## 3. Get the data

The recordings come from [FLEURS](https://huggingface.co/datasets/google/fleurs):
people reading Wikipedia sentences, with the correct transcript of each.

If the data is already prepared, for example with `sbatch prepare_data.sh` in
the repository root, it is used as it is. Otherwise `prepare_fleurs` downloads
the chosen split (a few hundred MB for the test split, a few minutes) and saves
it in the same format and folder as `prepare_data.py` does. So the notebook,
the batch job `aitta/transcribe_all.sh` and the other scripts all use the same
data, and it only needs to be downloaded once.

Each entry gets these columns:

| Column | Contents |
|---|---|
| `id` | a unique name for the recording, e.g. `test-000002-1736` |
| `audio` | the recording as a complete WAV file |
| `text` | the correct transcript |
| `language` | `fi` or `en` |
| `duration` | length in seconds |

In [ ]:
FLEURS_CONFIG = {"fi": "fi_fi", "en": "en_us"}   # FLEURS's names for the languages


def prepare_fleurs(language, split, out_dir):
    """Download one FLEURS split and save it like prepare_data.py does."""
    # FLEURS is stored on the Hub as parquet files under <language>/<split>/
    revision = "refs/convert/parquet"
    config = FLEURS_CONFIG[language]
    files = sorted(f for f in HfApi().list_repo_files("google/fleurs", repo_type="dataset", revision=revision)
                   if f.startswith(f"{config}/{split}/") and f.endswith(".parquet"))
    print(f"Downloading {len(files)} file(s) of FLEURS {config}/{split} ...")
    local = snapshot_download("google/fleurs", repo_type="dataset", revision=revision,
                              allow_patterns=files, max_workers=8)

    rows, skipped, n = [], 0, -1
    for path in files:
        table = pq.read_table(os.path.join(local, path),
                              columns=["id", "audio", "transcription", "raw_transcription"])
        for row in table.to_pylist():
            n += 1
            text = " ".join((row["raw_transcription"] or row["transcription"] or "").split())
            audio = row["audio"]["bytes"]           # FLEURS audio is already a 16 kHz WAV file
            try:
                with wave.open(io.BytesIO(audio)) as w:
                    duration = w.getnframes() / w.getframerate()
            except Exception:
                skipped += 1
                continue
            # The same rules as prepare_data.py, so both give the same ids
            if not text or not 0.5 <= duration <= 30:
                skipped += 1
                continue
            rows.append({"id": f"{split}-{n:06d}-{row['id']}", "audio": audio, "text": text,
                         "language": language, "duration": duration})

    features = Features({"id": Value("string"), "audio": Value("binary"), "text": Value("string"),
                         "language": Value("string"), "duration": Value("float32")})
    Dataset.from_list(rows, features=features).save_to_disk(out_dir)
    print(f"Saved {len(rows)} recordings ({skipped} skipped) to {out_dir}")


split_dir = os.path.join(WORK, "data", DATA, SPLIT)
if os.path.isdir(split_dir):
    print(f"Using the data already prepared in {split_dir}")
else:
    prepare_fleurs(LANGUAGE, SPLIT, split_dir)

Opening the dataset is instant: each entry is read from disk only when it is used.

In [ ]:
ds = load_from_disk(split_dir)
print(f"{len(ds)} recordings in {DATA} ({SPLIT})")

# The same results folder as aitta/transcribe_all.py uses
OUT_DIR = os.path.join(WORK, "results", f"aitta-{MODEL.split('/')[-1]}-{DATA}-{SPLIT}")
RESULTS = os.path.join(OUT_DIR, "transcripts.jsonl")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Results go to {OUT_DIR}")

## 4. Look at and listen to one recording

In [ ]:
entry = 0                 # change this to look at other recordings

example = ds[entry]
print(f"Id       : {example['id']}")
print(f"Duration : {example['duration']:.1f} s")
print(f"Text     : {example['text']}")
Audio(example["audio"])   # a player for the recording

## 5. Connect to AITTA

The access token is read from the environment variable `AITTA_API_KEY` if it is
set; otherwise you are asked to type it. It is not shown or saved in the notebook.
`max_retries=5`: if a call fails temporarily, the client waits and tries again.

In [ ]:
token = os.environ.get("AITTA_API_KEY") or getpass("AITTA access token: ")
client = openai.OpenAI(api_key=token, base_url=AITTA_BASE_URL, max_retries=5)

## 6. Transcribe one recording

One call to the function is **one API call**. The `audio` column already holds
a complete WAV file, so it is sent as it is.

In [ ]:
def transcribe_one(entry):
    """Send one recording to the model and show the result next to the correct text."""
    example = ds[entry]
    result = client.audio.transcriptions.create(
        file=("audio.wav", example["audio"], "audio/wav"),
        model=MODEL,
        response_format="verbose_json",   # also returns the detected language
    )
    print(f"Recording : {example['id']} ({example['duration']:.1f} s)")
    print(f"Language  : {result.language}")
    print(f"Correct   : {example['text']}")
    print(f"Model     : {result.text.strip()}")


transcribe_one(entry)

**Look at:** where do the correct text and the model's transcript differ? Small
differences, such as one letter in a long word, are common in Finnish.

## 7. Transcribe many recordings

`transcribe_many` sends recordings one by one and saves each result to
`transcripts.jsonl` right away. Recordings already in the file are skipped, so
running it again continues where it stopped.

**The notebook transcribes at most 50 recordings**, even if you ask for more or
give `None`. A notebook stops when the Jupyter session ends, so long runs belong
in a batch job:

```bash
export AITTA_API_KEY=<your token>
sbatch aitta/transcribe_all.sh --language fi
```

In [ ]:
NOTEBOOK_LIMIT = 50   # more than this: use aitta/transcribe_all.sh


def transcribe_many(max_entries=10):
    """Transcribe the first max_entries recordings (never more than NOTEBOOK_LIMIT)."""
    if max_entries is None or max_entries > NOTEBOOK_LIMIT:
        print(f"The notebook transcribes at most {NOTEBOOK_LIMIT} recordings. "
              f"For all of them, submit aitta/transcribe_all.sh.")
        max_entries = NOTEBOOK_LIMIT
    n = min(max_entries, len(ds))

    done = set()
    if os.path.exists(RESULTS):
        with open(RESULTS, encoding="utf-8") as f:
            done = {json.loads(line)["id"] for line in f if line.strip()}
    todo = [i for i in range(n) if ds[i]["id"] not in done]
    print(f"{n} recordings selected, {n - len(todo)} already done, {len(todo)} API calls to make")

    failed, start = [], time.time()
    with open(RESULTS, "a", encoding="utf-8") as out:
        for k, i in enumerate(todo, 1):
            example = ds[i]
            try:
                result = client.audio.transcriptions.create(
                    file=("audio.wav", example["audio"], "audio/wav"),
                    model=MODEL, response_format="verbose_json")
            except Exception as e:
                failed.append(example["id"])
                print(f"  FAILED {example['id']}: {e}")
                continue
            out.write(json.dumps({"model": f"aitta-{MODEL.split('/')[-1]}", "id": example["id"],
                                  "reference": example["text"], "transcript": result.text.strip(),
                                  "language": result.language}, ensure_ascii=False) + "\n")
            out.flush()   # saved right away
            if k % 10 == 0 or k == len(todo):
                print(f"  {k}/{len(todo)} done ({time.time() - start:.0f} s)")

    print(f"{len(todo) - len(failed)} transcribed, {len(failed)} failed. Saved in {RESULTS}")
    if failed:
        print("Run the cell again to retry the failed ones.")
    return n


n = transcribe_many(10)   # start with 10; try e.g. 50 next

## 8. How the error rate is calculated

The **word error rate (WER)** compares the model's transcript with the correct
text in three steps:

1. **Normalize both texts:** lowercase, remove punctuation, collapse spaces. Only
   the words count, not how the model formats them.
2. **Count the edits:** the smallest number of word replacements, deletions and
   insertions that turn the correct text into the transcript.
3. **Divide** the edits by the number of words in the correct text.

The **character error rate (CER)** is the same with characters instead of words.
It matters for Finnish: one wrong letter in a long compound word makes the whole
word wrong in WER, but is only one wrong character in CER.

The batch scripts compute the same numbers with the `jiwer` library; here the
calculation is written out so you can see how it works.

In [ ]:
def normalize(text):
    """Lowercase, punctuation removed, spaces collapsed (the same as in the scripts)."""
    text = "".join(c if c.isalnum() or c.isspace() else " " for c in text.lower())
    return " ".join(text.split())


def edit_distance(ref, hyp):
    """Smallest number of replacements, deletions and insertions that turn
    the list ref into the list hyp (of words or of characters)."""
    prev = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        cur = [i] + [0] * len(hyp)
        for j, h in enumerate(hyp, 1):
            cur[j] = min(prev[j] + 1,              # deletion
                         cur[j - 1] + 1,           # insertion
                         prev[j - 1] + (r != h))   # replacement (or match)
        prev = cur
    return prev[-1]


def wer_cer(references, hypotheses):
    """WER and CER over all sentences together."""
    word_errors = words = char_errors = chars = 0
    for ref, hyp in zip(references, hypotheses):
        ref, hyp = normalize(ref), normalize(hyp)
        word_errors += edit_distance(ref.split(), hyp.split())
        words += len(ref.split())
        char_errors += edit_distance(list(ref), list(hyp))
        chars += len(ref)
    return {"wer": word_errors / max(words, 1), "cer": char_errors / max(chars, 1)}


# Try it, and change the sentences to see how the numbers react
correct = "Miljoonia gallonia polttoainetta säästyy vuosittain."
model   = "miljoonia kalonia polttoainetta säästyy vuosittain"
print("Normalized correct :", normalize(correct))
print("Normalized model   :", normalize(model))
scores = wer_cer([correct], [model])
print(f"WER {scores['wer']:.0%}   CER {scores['cer']:.0%}")

## 9. Score the transcripts

The results are read into a table, scored, and saved as `side_by_side.tsv`,
which opens in Excel or LibreOffice: one row per sentence with the correct text,
the transcript and that sentence's WER.

In [ ]:
df = pd.read_json(RESULTS, lines=True)
first_n = set(ds.select(range(n))["id"])
df = df[df["id"].isin(first_n)].sort_values("id").reset_index(drop=True)

df["wer"] = [wer_cer([r], [h])["wer"] for r, h in zip(df["reference"], df["transcript"])]
total = wer_cer(df["reference"], df["transcript"])
print(f"{len(df)} sentences:  WER {total['wer']:.2%}   CER {total['cer']:.2%}")
print(f"Sentences without errors: {(df['wer'] == 0).mean():.0%}")

df.to_csv(os.path.join(OUT_DIR, "side_by_side.tsv"), sep="\t", index=False, encoding="utf-8-sig")
print(f"Saved {os.path.join(OUT_DIR, 'side_by_side.tsv')}")

Sorting by WER shows the sentences the model got most wrong. A sentence's WER can
be **over 100%**: added words count as errors too.

In [ ]:
pd.set_option("display.max_colwidth", None)     # show whole sentences
df.sort_values("wer", ascending=False)[["id", "wer", "language", "reference", "transcript"]].head(10)

## 10. Listen to the sentence with the most errors

Hearing the recording often explains the mistake: unclear speech, a foreign name,
background noise, or even an error in the correct text.

In [ ]:
worst = df.sort_values("wer", ascending=False).iloc[0]
print(f"{worst['id']}  (WER {worst['wer']:.0%})")
print(f"Correct : {worst['reference']}")
print(f"Model   : {worst['transcript']}")
Audio(ds[list(ds["id"]).index(worst["id"])]["audio"])

## Summary and next step

- One recording is one API call; results are saved as they arrive.
- WER counts word errors after normalizing both texts; CER does the same for characters.
- To transcribe **all** recordings, submit the batch job from the repository root:

```bash
export AITTA_API_KEY=<your token>
sbatch aitta/transcribe_all.sh --language fi
```

It continues from what this notebook already saved and writes `summary.md` and
`side_by_side.tsv` to the same folder.

**Note:** the audio is sent to an external service. That is fine for public data
like FLEURS, but check AITTA's terms before sending your own or sensitive recordings.